In [ ]:
# Uncomment if needed
# !pip install torch torchvision Pillow numpy matplotlib

In [ ]:
# Imports
import sys
import os

import numpy as np

import torch
import torch.nn as nn

In [ ]:
from autoencoder import SpatialVAE

In [ ]:
data_file = "./data_96_bw.npz"
output_file = "./latents.npz"
model_file = "./spatial_autoencoder.pth"

In [ ]:
# load data
data = np.load(data_file)

train_tensor = torch.from_numpy(data["train"].astype(np.float32))
test_tensor = torch.from_numpy(data["test"].astype(np.float32))

train_tensor_dataset = TensorDataset(
    train_tensor, torch.zeros(len(train_tensor), dtype=torch.long)
)
test_tensor_dataset = TensorDataset(
    test_tensor, torch.zeros(len(test_tensor), dtype=torch.long)
)

train_loader = DataLoader(
    train_tensor_dataset, batch_size=64, shuffle=True, num_workers=2
)
test_loader = DataLoader(
    test_tensor_dataset, batch_size=64, shuffle=False, num_workers=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
autoencoder = torch.load(model_file, weights_only=False)
autoencoder.eval()

all_latents = []

with torch.no_grad():
    for images, _ in train_loader:
        images = images.to(device)
        mean, _ = autoencoder.encode(images)
        all_latents.append(mean.cpu())
    for images, _ in test_loader:
        images = images.to(device)
        mean, _ = autoencoder.encode(images)
        all_latents.append(mean.cpu())

all_latents = torch.cat(all_latents, dim=0)
print(f"Latent tensor shape: {all_latents.shape}")
print(f"Uncompressed size: {all_latents.nbytes / 1e6:.1f} MB")

latents_means_mean = all_latents.mean(dim=0, keepdim=True)
latents_means_std  = all_latents.std(dim=0,  keepdim=True) + 1e-6

print(f"Latent mean range: [{latents_means_mean.min():.4f}, {latents_means_mean.max():.4f}]")
print(f"Latent std  range: [{latents_means_std.min():.4f},  {latents_means_std.max():.4f}]")

np.savez_compressed(
    output_file,
    latents=all_latents.numpy().astype(np.float16),
    latents_mean=latents_means_mean.numpy().astype(np.float16),
    latents_std=latents_means_std.numpy().astype(np.float16),
)

print("Saved.")